# Topic 18 — Anomaly Detection

> **Goal of this notebook:** learn to find the rare, unusual points in data — outliers and anomalies — using the main approaches (statistical, density, isolation, one-class), compare them, and build a from-scratch detector.

**Contents**
1. Concept & intuition
2. The approaches (statistical, density, isolation, one-class)
3. Key methods at a glance
4. The data: blobs with injected outliers
5. Isolation Forest
6. Comparing Isolation Forest, LOF & One-Class SVM
7. A Gaussian (Mahalanobis) detector from scratch
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

**Anomaly detection** flags observations that differ markedly from the majority — fraud, defects, intrusions, sensor faults. Anomalies are by definition **rare** and often **unlabelled**, so this is usually an unsupervised problem: model what "normal" looks like, then flag whatever doesn't fit.

Two related framings:
- **Outlier detection:** the training data already contains anomalies; find them.
- **Novelty detection:** train on clean data, then flag new points that look abnormal.

The central difficulty is the **class imbalance** — anomalies may be <1% of data — so accuracy is meaningless; we care about catching them (recall) without too many false alarms (precision).

## 2. The Approaches

| Approach | Idea |
|----------|------|
| **Statistical** | Fit a distribution (e.g. Gaussian); points with very low probability / high distance are anomalies. |
| **Density-based** | Points in low-density regions are anomalies (e.g. **Local Outlier Factor**). |
| **Isolation-based** | Anomalies are *easy to isolate* with random splits (**Isolation Forest**). |
| **One-class** | Learn a boundary around the normal data (**One-Class SVM**). |

## 3. Key Methods at a Glance

- **Isolation Forest** — builds random trees; anomalies need fewer splits to isolate (shorter paths). Fast, scales well, great default.
- **Local Outlier Factor (LOF)** — compares a point's local density to its neighbours'; works well when density varies.
- **One-Class SVM** — fits a tight boundary enclosing normal points; flexible but slower and sensitive to parameters.
- **Gaussian / Mahalanobis** — assumes normal data is ellipsoidal; flags points far from the mean in covariance-adjusted distance.

Most expose a **`contamination`** parameter — the expected fraction of anomalies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Data: Normal Cluster + Injected Outliers

We generate a dense normal cluster and scatter a few outliers around it. Because *we* inject them, we have ground-truth labels to score the detectors (using sklearn's convention: **+1 = normal, −1 = anomaly**).

In [ ]:
n_normal, n_out = 480, 20
normal = np.random.randn(n_normal, 2) * 1.0
outliers = np.random.uniform(low=-6, high=6, size=(n_out, 2))
X = np.vstack([normal, outliers])
y_true = np.r_[np.ones(n_normal), -np.ones(n_out)]   # +1 normal, -1 anomaly
contamination = n_out / (n_normal + n_out)

plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='coolwarm', s=15, edgecolor='k')
plt.title(f'Data: {n_normal} normal + {n_out} outliers'); plt.show()

## 5. Isolation Forest

In [ ]:
iso = IsolationForest(contamination=contamination, random_state=42)
y_pred = iso.fit_predict(X)
print(classification_report(y_true, y_pred, target_names=['anomaly (-1)', 'normal (+1)']))

# anomaly score (higher = more normal); ROC-AUC vs ground truth
scores = iso.score_samples(X)
print('ROC-AUC (Isolation Forest):', round(roc_auc_score(y_true, scores), 3))

## 6. Comparing Isolation Forest, LOF & One-Class SVM

In [ ]:
detectors = {
    'Isolation Forest': IsolationForest(contamination=contamination, random_state=42),
    'Local Outlier Factor': LocalOutlierFactor(contamination=contamination),
    'One-Class SVM': OneClassSVM(nu=contamination, gamma='scale'),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, det) in zip(axes, detectors.items()):
    pred = det.fit_predict(X)
    n_err = (pred != y_true).sum()
    ax.scatter(X[:, 0], X[:, 1], c=pred, cmap='coolwarm', s=15, edgecolor='k')
    ax.set_title(f'{name}\nmisclassified: {n_err}')
plt.show()

## 7. A Gaussian (Mahalanobis) Detector From Scratch

Assume normal data is roughly ellipsoidal. Fit its mean and covariance, then flag points whose **Mahalanobis distance** is in the top `contamination` fraction.

$$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$

In [ ]:
mu = X.mean(axis=0)
cov = np.cov(X, rowvar=False)
inv_cov = np.linalg.inv(cov)

diff = X - mu
mahal = np.sqrt(np.einsum('ij,jk,ik->i', diff, inv_cov, diff))

threshold = np.quantile(mahal, 1 - contamination)
pred_scratch = np.where(mahal > threshold, -1, 1)

print('From-scratch ROC-AUC:', round(roc_auc_score(y_true, -mahal), 3))
print(classification_report(y_true, pred_scratch, target_names=['anomaly (-1)', 'normal (+1)']))

## 8. Pros, Cons & When to Use

**Pros**
- Finds rare events without needing labelled anomalies.
- Isolation Forest is fast, scalable, and a strong default.
- A whole family of methods to match different data shapes/densities.

**Cons**
- Hard to evaluate without labels; must set a **contamination/threshold**.
- Methods make different assumptions (Gaussian shape, density, isolation) → results vary.
- High false-positive/negative trade-offs; domain knowledge is essential.

**When to use**
- Fraud, intrusion, fault, and quality-control detection.
- Any setting with rare, unlabelled abnormal events — or as a data-cleaning step to remove outliers before modelling.

## 9. Summary

- Anomaly detection models **"normal"** and flags points that don't fit — usually unsupervised and highly imbalanced, so judge it by recall/precision/ROC-AUC, not accuracy.
- The main families are **statistical**, **density** (LOF), **isolation** (Isolation Forest), and **one-class** (One-Class SVM).
- On injected outliers, all three library detectors recovered most anomalies, and our from-scratch **Mahalanobis** detector achieved a high ROC-AUC — confirming the statistical approach.
- This completes the course's tour of 18 core ML algorithms.